# Somina — Supplier Report

Pulls WooCommerce sales from MySQL, pulls the PRECIO price list from Google Sheets,
joins them on SKU, and exports an Excel report with cost and margin per product.

**The pipeline has three stages:**

| Stage | What happens | Sections |
|---|---|---|
| **Extract** | Read from MySQL and from Google Sheets | 3–5 |
| **Transform** | Clean, normalize SKUs, join, compute margin | 6–8 |
| **Load** | Write the Excel workbook | 10 |

Set the month in **Section 2** and run every cell from top to bottom.

## 1. Setup

Five libraries, each with one job:

- `pymysql` — the driver that actually speaks the MySQL protocol
- `sqlalchemy` — gives pandas a uniform connection object, and manages the connection pool
- `requests` — HTTP calls to the Apps Script endpoint
- `python-dotenv` — reads credentials from `.env` so they never appear in this file
- `openpyxl` — writes the `.xlsx`, including live formulas

Run the install cell once, then never again.

In [46]:
# Run once, then comment out
# !pip install pymysql sqlalchemy pandas python-dotenv openpyxl requests pyarrow

In [47]:
import os
import logging
from calendar import monthrange
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv
from sqlalchemy import create_engine, text, bindparam
from openpyxl.styles import Font
from openpyxl.utils import get_column_letter

load_dotenv(override=True)

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
pd.set_option("display.max_columns", 50)

logging.basicConfig(level=logging.INFO, format="%(levelname)-7s %(message)s", force=True)
log = logging.getLogger("somina")

## 2. Configuration — the only cell you edit

Everything downstream reads from here. This matters more than it looks: when dates are
defined in two different cells, whichever ran last silently wins, and you get a report
for a month you didn't ask for.

`monthrange` computes the last day, so February gets 28 or 29 without you thinking about it.

**`EXCLUDED_STATUSES`** is not an arbitrary list. It's what WooCommerce Analytics itself
leaves out, from `DataStore::get_excluded_report_order_statuses()` plus the `wc-` prefix
that `normalize_order_status()` adds. Note what is *not* excluded: `wc-refunded`,
`wc-processing` and `wc-on-hold` all count toward the dashboard's figures.

In [48]:
MONTH = "2026-07" # YYYY-MM — the only thing to change

EXCLUDED_STATUSES = ["wc-auto-draft", "wc-trash", "wc-pending", "wc-failed", "wc-cancelled"]

PRICE_COLS = ["COSTO", "PRECIO MAYORISTA", "TECNICO", "PRECIO FINAL SIN IVA"]

year, mon = (int(p) for p in MONTH.split("-"))
last_day  = monthrange(year, mon)[1]

START_DATE = f"{year}-{mon:02d}-01 00:00:00"
END_DATE   = f"{year}-{mon:02d}-{last_day} 23:59:59"

print(f"Periodo: {START_DATE}  ->  {END_DATE}")

Periodo: 2026-07-01 00:00:00  ->  2026-07-31 23:59:59


## 3. Connect to MySQL

Two settings that weren't in the first version of this notebook, and both fix a real problem:

**`pool_pre_ping=True`** sends a tiny `SELECT 1` before handing you a connection. Hostinger
closes idle connections after a few minutes; without this, the first query after a coffee
break fails with `PendingRollbackError`.

**`pool_recycle=280`** retires any connection older than 280 seconds, so SQLAlchemy replaces
it before the server kills it.

If you run this from your own machine rather than on the server, enable **Remote MySQL**
in hPanel and allow your IP. Also remember the host is `srv*.hstgr.io`, not the
`auth-db*.hstgr.io` address you use for phpMyAdmin.

In [ ]:
required = ["DB_USER", "DB_PASSWORD", "DB_HOST", "DB_NAME"]
missing = [k for k in required if not os.getenv(k)]
if missing:
    raise RuntimeError(f"Falta en .env: {', '.join(missing)}")

engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT', '3306')}/{os.getenv('DB_NAME')}"
    "?charset=utf8mb4",
    pool_pre_ping=True,
    pool_recycle=280,
)

with engine.connect() as conn:
    print(conn.execute(text("SELECT VERSION(), DATABASE()")).fetchone())

## 4. Extract — sales from WooCommerce

This is the query behind the **Productos** report, reassembled from
`src/Admin/API/Reports/Products/DataStore.php`.

**Why `wp_wc_order_product_lookup` and not the raw order items?** WooCommerce maintains this
table as a flat, pre-aggregated cache — one row per line item with revenue already computed.
Rebuilding the same numbers from `wp_postmeta` takes three joins and a pivot. The trade-off
is that it *is* a cache, so it can drift; the reconciliation notebook is where you check that.

**Three details that are easy to get wrong:**

- The lookup table has no `status` column, so it joins back to `wp_wc_order_stats` just to
  filter. Querying it standalone silently includes cancelled and failed orders.
- Refunds are stored as rows with negative `product_qty`, so `SUM` nets them out for free.
  The `>= 0` inside `orders` stops those refund rows from counting as extra orders.
- SKU comes from `wp_wc_product_meta_lookup`, where `product_id` is the primary key. The
  double join handles variations: a variation's own SKU wins, and it falls back to the
  parent's when the variation has none.

In [ ]:
SALES_SQL = """
SELECT
    pl.product_id,
    pl.variation_id,
    COALESCE(NULLIF(pmlv.sku, ''), pmlp.sku) AS sku,
    p.post_title                             AS product_name,
    SUM(pl.product_qty)                      AS items_sold,
    SUM(pl.product_net_revenue)              AS net_sales,
    COUNT(DISTINCT CASE WHEN pl.product_gross_revenue >= 0
                        THEN pl.order_id END) AS orders
FROM wp_wc_order_product_lookup pl
JOIN wp_wc_order_stats os
  ON os.order_id = pl.order_id
LEFT JOIN wp_posts p
  ON p.ID = pl.product_id
LEFT JOIN wp_wc_product_meta_lookup pmlv
  ON pmlv.product_id = pl.variation_id
LEFT JOIN wp_wc_product_meta_lookup pmlp
  ON pmlp.product_id = pl.product_id
WHERE os.status NOT IN :excluded
  AND pl.date_created >= :start
  AND pl.date_created <= :end
GROUP BY pl.product_id, pl.variation_id, sku, p.post_title
ORDER BY net_sales DESC
"""

stmt = text(SALES_SQL).bindparams(
    bindparam("excluded", value=EXCLUDED_STATUSES, expanding=True),
    start=START_DATE,
    end=END_DATE,
)

df_sales = pd.read_sql(stmt, engine)

print(f"{len(df_sales)} productos | {df_sales.items_sold.sum():,.0f} unidades | "
      f"{df_sales.net_sales.sum():,.0f} en ventas netas")
df_sales.head(10)

50 productos | 1,238 unidades | 12,644,023 en ventas netas


,product_id,variation_id,sku,product_name,items_sold,net_sales,orders
0,1096,0,ACBC,Acople Borden Chino- Acople Borden Importado,103.00,"1,405,223.00",8
1,355,0,HATH,Holder Automático tipo Huevo,81.00,"1,320,000.00",10
2,352,0,VR,Valvula Reguladora,69.00,"1,113,400.00",9
3,372,0,PMN,Pedal Neumatico Odontologico,21.00,"1,008,000.00",6
4,356,0,VCV,Valvula Cambia Vias,38.00,"768,050.00",7
5,1870,0,VRA,"Reguladora de aire con filtro, Valvula Regulad...",15.00,"696,000.00",3
6,379,0,VOF,Valvula On/Off,31.00,"592,800.00",4
7,353,0,JT,Jeringa triple,17.00,"549,600.00",10
8,392,0,MPM,Manguera Pieza Mano X metro,58.00,"452,400.00",1
9,389,0,MN14A,Manguera 1/4 Amarilla X metro,107.00,"417,700.00",8


## 5. Extract — prices from Google Sheets

The PRICE list lives in a private sheet, so it can't be fetched as a public CSV. The Apps
Script web app solves that: it runs under your own Google identity (`Execute as: Me`), so it
can read the sheet, and the shared token is what stops anyone with the URL from reading it too.

**The `Content-Type` check is not optional.** When `doGet` throws, Apps Script returns an HTML
error page with status 200. Without this check, `resp.json()` fails with a parse error that
tells you nothing about what actually broke.

In [ ]:
url   = os.getenv("APPS_SCRIPT_URL")
token = os.getenv("APPS_SCRIPT_TOKEN")

if not url or not token:
    raise RuntimeError("Falta APPS_SCRIPT_URL o APPS_SCRIPT_TOKEN en .env")

resp = requests.get(url, params={"token": token, "sheet": "PRECIO"}, timeout=30)
resp.raise_for_status()

if "json" not in resp.headers.get("Content-Type", ""):
    raise RuntimeError(f"Apps Script devolvio {resp.headers.get('Content-Type')}:\n{resp.text[:300]}")

data = resp.json()
if isinstance(data, dict):
    if "error" in data:
        raise RuntimeError(f"Apps Script: {data['error']}")
    data = data.get("rows", data)

df_precios_raw = pd.DataFrame(data)

print(f"{len(df_precios_raw)} filas")
print(df_precios_raw.columns.tolist())
df_precios_raw.head()

112 filas
['SKU', 'PRODUCTOS', 'COSTO', 'PRECIO MAYORISTA', 'MARGEN MAYORISTA', 'TECNICO', 'PRECIO WOOCOMMERCE', 'PROVEEDOR', 'CATEGORIA', 'ID PRODUCTO', '']


,SKU,PRODUCTOS,COSTO,PRECIO MAYORISTA,MARGEN MAYORISTA,TECNICO,PRECIO WOOCOMMERCE,PROVEEDOR,CATEGORIA,ID PRODUCTO,
0,KBEC,Buje Escupidera chino,16500,22000,0.25,23200,30200,MIGUEL,PARTE UNIDENTAL,382,
1,FAR,Filtro de aire,12000,18000,0.33,18800,24400,PNEUMATIC,VALVULA,4376,
2,BPUH3,Bombillo Halogeno o Lampara Halogena,6000,10000,0.40,10400,13500,ALIBA,PARTE UNIDENTAL,1581,
3,KPC,Punta Caucho Set Eyector o Punta Boquilla de E...,4500,8000,0.44,8300,10800,MIGUEL,REPUESTO,1880,
4,PUDI,Pedal Unidad Dental- Pedal Odontologico Importado,35000,52000,0.33,54500,70900,PNEUMATIC,PARTE UNIDENTAL,2900,


## 6. Transform — clean the price list

Three problems arrive with this data, and all three are silent:

**The SKU column has no header.** The top-left cell in PRECIO is empty, so `JSON.stringify`
gives it the key `""`. Rename it before anything else — and note that any code doing
`df.columns != ""` to drop blank columns would delete exactly the column you need.

**`"PAPELERA"` sits where prices should be.** Discontinued items, presumably. One text value
in a column forces the whole column to `object` dtype, which breaks arithmetic. `errors="coerce"`
turns them into `NaN`, which pandas handles correctly.

**Duplicate SKUs.** Excel's VLOOKUP silently takes the first match. Pandas `.merge()` does
something worse: it duplicates your sales rows, one per matching price row, so quantities and
totals double with no warning. Deduplicate here, before the join.

In [ ]:
df_precios = df_precios_raw.copy()
df_precios.columns = [str(c).strip() for c in df_precios.columns]

# Find the SKU column whatever its capitalization; fall back to the old blank header
sku_col = next((c for c in df_precios.columns if c.upper() == "SKU"), None)
if sku_col is None and "" in df_precios.columns:
    sku_col = ""
if sku_col is None:
    raise RuntimeError(f"No hay columna SKU. Encontradas: {df_precios.columns.tolist()}")

df_precios = df_precios.rename(columns={sku_col: "sku", "ID PRODUCTO": "product_id_precio"})
df_precios = df_precios.loc[:, df_precios.columns != ""]      # trailing empty column

df_precios["sku"] = (df_precios["sku"].astype(str).str.strip().str.upper()
                     .replace({"NAN": pd.NA, "": pd.NA}))
df_precios = df_precios.dropna(subset=["sku"])

for c in (c for c in PRICE_COLS + ["MARGEN MAYORISTA"] if c in df_precios.columns):
    df_precios[c] = pd.to_numeric(df_precios[c], errors="coerce")

if "product_id_precio" in df_precios.columns:
    df_precios["product_id_precio"] = pd.to_numeric(
        df_precios["product_id_precio"], errors="coerce").astype("Int64")

for c in ("PROVEEDOR", "CATEGORIA"):
    if c in df_precios.columns:
        df_precios[c] = df_precios[c].astype(str).str.strip().replace({"": pd.NA})

dupes = int(df_precios["sku"].duplicated().sum())
if dupes:
    log.warning("%d SKU duplicados en PRECIO — se conserva el primero", dupes)
    df_precios = df_precios.drop_duplicates(subset="sku", keep="first")

print(f"{len(df_precios)} SKU")
df_precios.head()

112 SKU


,sku,PRODUCTOS,COSTO,PRECIO MAYORISTA,MARGEN MAYORISTA,TECNICO,PRECIO WOOCOMMERCE,PROVEEDOR,CATEGORIA,product_id_precio
0,KBEC,Buje Escupidera chino,16500,22000,0.25,23200,30200,MIGUEL,PARTE UNIDENTAL,382
1,FAR,Filtro de aire,12000,18000,0.33,18800,24400,PNEUMATIC,VALVULA,4376
2,BPUH3,Bombillo Halogeno o Lampara Halogena,6000,10000,0.40,10400,13500,ALIBA,PARTE UNIDENTAL,1581
3,KPC,Punta Caucho Set Eyector o Punta Boquilla de E...,4500,8000,0.44,8300,10800,MIGUEL,REPUESTO,1880
4,PUDI,Pedal Unidad Dental- Pedal Odontologico Importado,35000,52000,0.33,54500,70900,PNEUMATIC,PARTE UNIDENTAL,2900


## 7. Transform — join on SKU

This is the whole point of the pipeline: two systems that share nothing except a product code.

**Normalizing both sides is what makes the join work.** Sheets and WooCommerce rarely format
SKUs identically — trailing spaces, lowercase, a numeric SKU that Sheets returns as `12345.0`
while WooCommerce stores `"12345"`. If your match rate looks low, this is the first thing to check.

**`validate="many_to_one"`** is the single most valuable line in the notebook. It tells pandas
that many sales rows may match one price row, but never the reverse — and raises if that's
violated. Without it, a duplicate in PRECIO silently inflates your totals.

**`how="left"`** keeps every product that sold, even when it has no price. Those rows become
your to-do list rather than disappearing.

In [ ]:
df = df_sales.copy()
df["sku"] = (df["sku"].astype(str).str.strip().str.upper()
             .replace({"NAN": pd.NA, "": pd.NA}))

keep = ["sku"] + [c for c in PRICE_COLS + ["PROVEEDOR"] if c in df_precios.columns]

df = df.merge(df_precios[keep], on="sku", how="left", validate="many_to_one")

df["costo_total"] = df["items_sold"] * df["COSTO"]
df["margen"]      = df["net_sales"] - df["costo_total"]
df["margen_pct"]  = df["margen"] / df["net_sales"].where(df["net_sales"] != 0) * 100

df.head(10)

,product_id,variation_id,sku,product_name,items_sold,net_sales,orders,COSTO,PRECIO MAYORISTA,TECNICO,PROVEEDOR,costo_total,margen,margen_pct
0,1096,0,ACBC,Acople Borden Chino- Acople Borden Importado,103.00,"1,405,223.00",8,7200,10200,10700,ALIBA,"741,600.00","663,623.00",47.23
1,355,0,HATH,Holder Automático tipo Huevo,81.00,"1,320,000.00",10,12000,16000,16800,MIGUEL,"972,000.00","348,000.00",26.36
2,352,0,VR,Valvula Reguladora,69.00,"1,113,400.00",9,12500,15800,16700,MIGUEL,"862,500.00","250,900.00",22.53
3,372,0,PMN,Pedal Neumatico Odontologico,21.00,"1,008,000.00",6,32500,47000,49300,MIGUEL,"682,500.00","325,500.00",32.29
4,356,0,VCV,Valvula Cambia Vias,38.00,"768,050.00",7,16200,20000,21100,MIGUEL,"615,600.00","152,450.00",19.85
5,1870,0,VRA,"Reguladora de aire con filtro, Valvula Regulad...",15.00,"696,000.00",3,32000,46000,48200,FABIA BORDA,"480,000.00","216,000.00",31.03
6,379,0,VOF,Valvula On/Off,31.00,"592,800.00",4,15500,19000,20100,MIGUEL,"480,500.00","112,300.00",18.94
7,353,0,JT,Jeringa triple,17.00,"549,600.00",10,27000,31000,32900,ALIBA,"459,000.00","90,600.00",16.48
8,392,0,MPM,Manguera Pieza Mano X metro,58.00,"452,400.00",1,5500,7800,8200,MARCO,"319,000.00","133,400.00",29.49
9,389,0,MN14A,Manguera 1/4 Amarilla X metro,107.00,"417,700.00",8,2800,3900,4100,MARCO,"299,600.00","118,100.00",28.27


## 8. Validate — surface what would otherwise pass silently

`NaN` is contagious in one direction and invisible in the other. `NaN * 5` is `NaN`, but
`Series.sum()` **skips** `NaN` instead of propagating it. So a product with no cost in PRECIO
doesn't break anything — it just makes your total cost lower and your margin better than reality,
with no error and no warning.

This cell turns that into something you can see.

In [ ]:
sin_sku   = int(df["sku"].isna().sum())
sin_costo = int(df["COSTO"].isna().sum())

print(f"Productos          : {len(df)}")
print(f"Sin SKU            : {sin_sku}")
print(f"Sin costo en PRECIO: {sin_costo} ({sin_costo / len(df) * 100:.0f}%)")
print(f"Unidades           : {df.items_sold.sum():,.0f}")
print(f"Ventas netas       : {df.net_sales.sum():,.0f}")
print(f"Costo              : {df.costo_total.sum():,.0f}")
print(f"Margen             : {df.margen.sum():,.0f}")

if sin_costo:
    print("\nProductos vendidos sin costo registrado:")
    display(df[df["COSTO"].isna()][["sku", "product_name", "items_sold", "net_sales"]])

Productos          : 50
Sin SKU            : 0
Sin costo en PRECIO: 0 (0%)
Unidades           : 1,238
Ventas netas       : 12,644,023
Costo              : 9,209,500
Margen             : 3,434,523


## 9. Explore

This is what a notebook is actually good at, and why the pipeline is worth keeping here rather
than only in a script. Slice it however the question requires.

In [ ]:
# Por proveedor
if "PROVEEDOR" in df.columns:
    resumen = df.groupby("PROVEEDOR", dropna=False).agg(
        productos=("sku", "nunique"),
        unidades=("items_sold", "sum"),
        ventas=("net_sales", "sum"),
        costo=("costo_total", "sum"),
        margen=("margen", "sum"),
    ).sort_values("ventas", ascending=False)
    display(resumen)

,productos,unidades,ventas,costo,margen
PROVEEDOR,,,,,
MIGUEL,13,463.00,"5,800,146.00","4,391,900.00","1,408,246.00"
MARCO,14,269.00,"2,877,400.00","2,106,400.00","771,000.00"
ALIBA,10,137.00,"2,323,797.00","1,483,800.00","839,997.00"
FABIA BORDA,10,361.00,"1,200,180.00","865,900.00","334,280.00"
PNEUMATIC,2,3.00,"416,000.00","344,000.00","72,000.00"
DANIEL,1,5.00,"26,500.00","17,500.00","9,000.00"


In [ ]:
# Un proveedor en concreto
PROVEEDOR = "MIGUEL"

prov = df[df["PROVEEDOR"] == PROVEEDOR]
print(f"{PROVEEDOR}: {prov['sku'].nunique()} productos, "
      f"{prov['items_sold'].sum():,.0f} unidades, "
      f"{prov['net_sales'].sum():,.0f} en ventas")

prov.sort_values("net_sales", ascending=False).head(30)

MIGUEL: 13 productos, 463 unidades, 5,800,146 en ventas


,product_id,variation_id,sku,product_name,items_sold,net_sales,orders,COSTO,PRECIO MAYORISTA,TECNICO,PROVEEDOR,costo_total,margen,margen_pct
1,355,0,HATH,Holder Automático tipo Huevo,81.00,"1,320,000.00",10,12000,16000,16800,MIGUEL,"972,000.00","348,000.00",26.36
2,352,0,VR,Valvula Reguladora,69.00,"1,113,400.00",9,12500,15800,16700,MIGUEL,"862,500.00","250,900.00",22.53
3,372,0,PMN,Pedal Neumatico Odontologico,21.00,"1,008,000.00",6,32500,47000,49300,MIGUEL,"682,500.00","325,500.00",32.29
4,356,0,VCV,Valvula Cambia Vias,38.00,"768,050.00",7,16200,20000,21100,MIGUEL,"615,600.00","152,450.00",19.85
6,379,0,VOF,Valvula On/Off,31.00,"592,800.00",4,15500,19000,20100,MIGUEL,"480,500.00","112,300.00",18.94
14,3219,0,VVB2,Valvula Ventury Baja Bronce,22.00,"350,800.00",7,13000,16500,17400,MIGUEL,"286,000.00","64,800.00",18.47
16,354,0,VP,Válvula Piloto,13.00,"230,500.00",7,14000,17500,18500,MIGUEL,"182,000.00","48,500.00",21.04
20,397,0,F14,Fitin Racor 1/4,152.00,"125,500.00",6,650,850,900,MIGUEL,"98,800.00","26,700.00",21.27
21,373,0,HSTH,Holder Sencillo Tipo Huevo,14.00,"98,700.00",2,5500,7000,7400,MIGUEL,"77,000.00","21,700.00",21.99
24,371,0,SE,Set Eyector o Boquilla eyectora,15.00,"78,500.00",6,3800,5000,5300,MIGUEL,"57,000.00","21,500.00",27.39


In [ ]:
# Top 20 por margen — y los que pierden dinero
display(df.nlargest(20, "margen")[["sku", "product_name", "items_sold", "net_sales", "margen", "margen_pct"]])

perdidas = df[df["margen"] < 0]
if len(perdidas):
    print(f"\n{len(perdidas)} productos vendidos por debajo del costo:")
    display(perdidas[["sku", "product_name", "items_sold", "COSTO", "net_sales", "margen"]])

,sku,product_name,items_sold,net_sales,margen,margen_pct
0,ACBC,Acople Borden Chino- Acople Borden Importado,103.00,"1,405,223.00","663,623.00",47.23
1,HATH,Holder Automático tipo Huevo,81.00,"1,320,000.00","348,000.00",26.36
3,PMN,Pedal Neumatico Odontologico,21.00,"1,008,000.00","325,500.00",32.29
2,VR,Valvula Reguladora,69.00,"1,113,400.00","250,900.00",22.53
5,VRA,"Reguladora de aire con filtro, Valvula Regulad...",15.00,"696,000.00","216,000.00",31.03
4,VCV,Valvula Cambia Vias,38.00,"768,050.00","152,450.00",19.85
8,MPM,Manguera Pieza Mano X metro,58.00,"452,400.00","133,400.00",29.49
9,MN14A,Manguera 1/4 Amarilla X metro,107.00,"417,700.00","118,100.00",28.27
6,VOF,Valvula On/Off,31.00,"592,800.00","112,300.00",18.94
10,RMJT,Rollo 50 mts Manguera Jeringa Triple,2.00,"410,000.00","110,000.00",26.83


## 10. Load — the Excel workbook

The stage your notebook was missing. Until the result leaves the kernel, this is an analysis,
not a pipeline.

**`=SUM()` instead of a computed number.** If anyone edits a row in Excel, a hardcoded total
goes stale and nobody notices. A formula recalculates.

**The `[:31]` on sheet names** isn't arbitrary — Excel rejects names longer than 31 characters,
and also `: \ / ? * [ ]`.

In [ ]:
SHEET_COLS = ["sku", "product_name", "items_sold", "COSTO",
              "costo_total", "net_sales", "margen", "margen_pct"]
MONEY_COLS = ["COSTO", "costo_total", "net_sales", "margen"]


def write_sheet(writer, data, name):
    cols = [c for c in SHEET_COLS if c in data.columns]
    data = data[cols].sort_values("net_sales", ascending=False)
    data.to_excel(writer, sheet_name=name, index=False)

    ws = writer.sheets[name]
    last, total_row = len(data) + 1, len(data) + 2

    ws.cell(row=total_row, column=1, value="TOTAL").font = Font(bold=True)
    for col in ["items_sold", "costo_total", "net_sales", "margen"]:
        if col in cols:
            i = cols.index(col) + 1
            letter = get_column_letter(i)
            ws.cell(row=total_row, column=i, value=f"=SUM({letter}2:{letter}{last})").font = Font(bold=True)

    for col in (c for c in MONEY_COLS if c in cols):
        letter = get_column_letter(cols.index(col) + 1)
        for cell in ws[letter][1:]:
            cell.number_format = "$#,##0"

    for i, col in enumerate(cols, start=1):
        width = max(len(col), int(data[col].astype(str).str.len().max() or 0)) + 2
        ws.column_dimensions[get_column_letter(i)].width = min(width, 45)

    ws.freeze_panes = "A2"


def safe_name(v):
    if pd.isna(v):
        return "SIN PROVEEDOR"
    name = str(v)
    for bad in ":\\/?*[]":
        name = name.replace(bad, "-")
    return name[:31] or "SIN NOMBRE"


Path("output").mkdir(exist_ok=True)
path = Path("output") / f"reporte_proveedores_{MONTH}.xlsx"

with pd.ExcelWriter(path, engine="openpyxl") as writer:
    write_sheet(writer, df, "TODOS")
    if "PROVEEDOR" in df.columns:
        for prov_name, grupo in df.groupby("PROVEEDOR", dropna=False):
            write_sheet(writer, grupo, safe_name(prov_name))

print("Guardado:", path.resolve())

Guardado: /Users/miguelangelduarte/Desktop/PIPELINE FOR WOOCOMERCES/PANDAS/output/reporte_proveedores_2026-09.xlsx


In [ ]:
engine.dispose()
print("Conexion cerrada")

Conexion cerrada


## 11. Where this goes next

**Split the reconciliation out.** Your other queries — the ones comparing `wp_wc_order_stats`
against raw `wp_postmeta` — answer a different question with a different lifespan. They deserve
their own notebook, and that's the one worth putting in the repo.

**Move the pipeline into a module.** Once the steps above stop changing, the extract/transform/load
functions belong in `somina_etl.py`, imported here in one line. The notebook then keeps only
Section 9 — the exploring, which is the part that genuinely needs to be interactive.

**Then schedule it.** A module with a `--month` argument runs from cron on the first of the month
and emails the workbook. That's the difference between a report you run and a report that arrives.

**One thing to check before trusting any of this.** If the totals here don't match the WooCommerce
dashboard, look at **Analytics → Settings → Date type** before assuming the data is wrong.
Filtering on *date created* while the dashboard is set to *date paid* produces a real difference
with nothing broken on either side.